# Batch_03 EDA dossier

Section-by-section analysis of the batch_03 data. Each section states
what is measured and **which design decision it feeds**; a Findings
stub at the end of each section collects the conclusions that flow
back into the design's EDA-gated open questions.

Prerequisite read: `00_batch03_overview.ipynb` (sources, schemas,
entity diagram).

## 0. What this data cannot answer

- **No labels** — attribution/conversion data is not in batch_03; no
  CVR, class balance, or label-volume analysis here.
- **No user-level signal** — user/device identifiers are anonymized;
  user features cannot be assessed.
- **No loss side** — the firehose sample is wins-only by construction;
  nothing here describes lost or non-bid auctions.
- **No volume representativity** — 933 events; volumes come from the
  traffic CSVs (section 6), not from the sample.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from eda_tools import (
    TRAFFIC_DIR,
    load_activity,
    load_candidates,
    load_delivery_control,
    load_firehose,
)

pd.set_option("display.max_columns", 200)

events = load_firehose()
candidates = load_candidates()
activity = load_activity()
entities = load_delivery_control()

## 1. Feature availability at bid time

**Feeds: the candidate feature list and the GBDT input schema**
(request-derived features are the model's main input; this section is
the reason we asked for samples).

- Coverage (non-null %) of every bid-time field: `app.bundle/name/
  publisher`, `device.devicetype/os`, `geo.country` (+ finer geo if
  present), `imp[].video` specs (duration, placement, mimes, skip),
  `imp[].bidfloor`, PMP/deal presence, `cur`, `tmax`,
  `inventory_source`.
- Cardinality + top-k frequencies per categorical; concentration (mass
  in the top 10 bundles / geos).
- Cross-file consistency: field presence per `source_file`,
  `data_center` values, timestamp range per file (regions? days?).

Output: one coverage/cardinality table ranked for feature candidacy.

**Findings (1):** _to fill_

## 2. Join & identity dossier

**Feeds: label-join readiness** (the join the whole label build
stands on) and the identity-triplet requirement.

- Key integrity: `auction_key` uniqueness; `requestid_str` uniqueness;
  duplicate events across the three files.
- Standing check: winner-in-candidates (933/933 at ingest).
- Id spaces: DSP integer ids (candidates/impressions) vs `ua_*` UUIDs
  (delivery control); bridge coverage (14 adsets / 121 line items) —
  what share of impression and candidate rows the export can enrich.
- Timestamp sanity: `rx_timestamp_usecs` ranges; impression-before-
  activity ordering violations.

**Findings (2):** _to fill_

## 3. Candidate structure within a request

**Feeds: what the model actually ranks** — separating candidate-
discriminating features from per-request context features.

- Candidates-per-request distribution (per file, per hour).
- Within-auction variance: which candidate fields differ inside one
  auction (line item, campaign, deal, bid) vs constant context (app,
  geo, device).
- `bidding.bid_price_micros` spread within and across auctions;
  `pace_delivery` share; deal presence rate.
- Winner vs losers inside each auction: systematic differences (bid,
  deal, pacing flag)?

**Findings (3):** _to fill_

## 4. Activity funnel

**Feeds: feature / proxy-metric candidacy** (the rollout design's
`proxy` evidence mode needs early signals).

- Quartile funnel START → Q1 → MID → Q3 (+ completion if present),
  overall and per app bundle / line item.
- Delay from impression `rx_timestamp_usecs` to each quartile — how
  early a "watched" proxy exists.
- Events-per-impression distribution; the 73 impressions with no
  activity — pattern or noise?

**Findings (4):** _to fill_

## 5. Delivery-control config landscape

**Feeds: context for the ad-set settings feed** (what config diversity
the agent will consume).

- Per entity type: distributions of status, budgets, `bid`,
  `objective_type`, frequency caps, flight dates.
- Raw `bidding.*` / `targeting` shapes on the 14 adsets; confirm the
  `bidding_strategy` / `pacing` null placeholders.
- Hierarchy integrity: every child's `ua_*_id` parent exists in the
  export.

**Findings (5):** _to fill_

## 6. Traffic & workload metrics (client CSVs)

**Feeds: capacity sizing and freshness budgets** with the
client-measured numbers (Oct 1–29).

Files in `TRAFFIC_DIR` (plain `pd.read_csv`):

- Traffic profile: auctions/bids/wins/cost per 15 min — daily and
  weekly seasonality; bid rate and win rate over time (whole-period
  aggregates 4.75% / 0.88%, wide per-interval variation).
- Impression-delay histogram: CDF with real percentiles (p50 ≈ 2.3
  min, p90 ≈ 24 min — the client prose said 15 min; the CSV is the
  authority).
- Active lines / campaigns per hour.
- RPS summary table (avg ~444k / p99 ~693k / max ~1.92M).

In [ ]:
sorted(p.name for p in TRAFFIC_DIR.glob("*.csv"))

**Findings (6):** _to fill_

## 7. Economics / bid shading

**Feeds: the max-bid-cap discussion and the win-rate / bid-landscape
research direction.**

- `bid_price_micros_usd` vs `win_price_micros_usd` distributions and
  ratio (shading depth); `bid_shading_info.*`; `exchange_bid_price`.
- Win price vs the losing candidates' `bidding.bid_price_micros` —
  the per-auction bid landscape.
- CPM sanity check against the traffic profile's ~$10.

**Findings (7):** _to fill_